# Training Regression - Reaction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemprop/chemprop/blob/main/examples/training_regression_reaction.ipynb)

In [ ]:
# # Install chemprop from GitHub if running in Google Colab
# import os

# if os.getenv("COLAB_RELEASE_TAG"):
#     try:
#         import chemprop
#     except ImportError:
#         !git clone https://github.com/chemprop/chemprop.git
#         %cd chemprop
#         !pip install .
#         %cd examples

# Import packages

In [ ]:
import pandas as pd
from lightning import pytorch as pl
from pathlib import Path

from chemprop import data, featurizers, models, nn

# Change data inputs here

In [ ]:
chemprop_dir = Path.cwd().parent
input_path = chemprop_dir / "tests" / "data" / "regression" / "rxn" / "e2sn2.csv"
num_workers = 0  # number of workers for dataloader. 0 means using main process for data loading
smiles_column = 'AAM'
target_columns = ['ea']

## Load data

In [ ]:
df_input = pd.read_csv(input_path)
df_input

## Load smiles and targets

In [ ]:
smis = df_input.loc[:, smiles_column].values
ys = df_input.loc[:, target_columns].values

smis[:5], ys[:5]

## Get datapoints

In [ ]:
all_data = [data.ReactionDatapoint.from_smi(smi, y) for smi, y in zip(smis, ys)]

## Perform data splitting for training, validation, and testing

In [ ]:
mols = [d.rct for d in all_data]  # Can either split by reactants (.rct) or products (.pdt)
train_indices, val_indices, test_indices = data.make_split_indices(mols, "random", (0.8, 0.1, 0.1))
train_data, val_data, test_data = data.split_data_by_indices(
    all_data, train_indices, val_indices, test_indices
)

In [ ]:
print(type(train_data))
print(train_data)

# Defining the featurizer

Reactions can be featurized using the ```CondensedGraphOfReactionFeaturizer``` (also labeled ```CGRFeaturizer```).


Use ```_mode``` keyword to set the mode by which a reaction should be featurized into a ```MolGraph```.

Options are can be found with ```featurizers.RxnMode.keys```

In [ ]:
for key in featurizers.RxnMode.keys():
    print(key)

In [ ]:
featurizer = featurizers.CondensedGraphOfReactionFeaturizer(mode_="REAC_DIFF")

## Get ReactionDatasets

In [ ]:
train_dset = data.ReactionDataset(train_data[0], featurizer)
scaler = train_dset.normalize_targets()

val_dset = data.ReactionDataset(val_data[0], featurizer)
val_dset.normalize_targets(scaler)
test_dset = data.ReactionDataset(test_data[0], featurizer)

## Get dataloaders

In [ ]:
train_loader = data.build_dataloader(train_dset, num_workers=num_workers)
val_loader = data.build_dataloader(val_dset, num_workers=num_workers, shuffle=False)
test_loader = data.build_dataloader(test_dset, num_workers=num_workers, shuffle=False)

# Change Message-Passing Neural Network (MPNN) inputs here

## Message passing

Message passing blocks must be given the shape of the featurizer's outputs.

Options are `mp = nn.BondMessagePassing()` or `mp = nn.AtomMessagePassing()`

In [ ]:
fdims = featurizer.shape # the dimensions of the featurizer, given as (atom_dims, bond_dims).
mp = nn.BondMessagePassing(*fdims)

## Aggregation

In [ ]:
print(nn.agg.AggregationRegistry)

In [ ]:
agg = nn.MeanAggregation()

## Feed-Forward Network (FFN)

In [ ]:
print(nn.PredictorRegistry)

In [ ]:
output_transform = nn.UnscaleTransform.from_standard_scaler(scaler)

In [ ]:
ffn = nn.RegressionFFN(output_transform=output_transform)

## Batch norm

In [ ]:
batch_norm = True

## Metrics

In [ ]:
print(nn.metrics.MetricRegistry)

In [ ]:
metric_list = [nn.metrics.RMSE(), nn.metrics.MAE()] 
# Only the first metric is used for training and early stopping

## Construct MPNN

In [ ]:
mpnn = models.MPNN(mp, agg, ffn, batch_norm, metric_list)
mpnn

# Training and testing

## Set up trainer

In [ ]:
trainer = pl.Trainer(
    logger=False,
    enable_checkpointing=True,  # Use `True` if you want to save model checkpoints. The checkpoints will be saved in the `checkpoints` folder.
    enable_progress_bar=True,
    accelerator="auto",
    devices=1,
    max_epochs=100,  # number of epochs to train for
)

## Start training

In [ ]:
trainer.fit(mpnn, train_loader, val_loader)

## Test results

In [ ]:
results = trainer.test(mpnn, test_loader)

In [ ]:
# import chemprop
# arguments = [
#     '--data_path', '../tests/data/regression/rxn/lograte.csv',
#     '--dataset_type', 'regression',
#     '--output-dir', 'train_example',
#     '--epochs', '100', 
#     '--reaction',
#     '--save_smiles_splits'
# ]

# args = chemprop.args.TrainArgs().parse_args[arguments]
# mean_score, std_score = chemprop.train.cross_validation(args=args, train_func=chemprop.train.run_training)

In [ ]:
# import chemprop
# arguments = [
#     '--data_path', '../tests/data/regression/rxn/lograte.csv',
#     '--t', 'regression',
#     '--output-dir', 'train_example',
#     '--epochs', '100', 
#     '--reaction',
#     '--save_smiles_splits'
# ]

# args = chemprop.train.add_args[arguments]
# mean_score, std_score = chemprop.train.cross_validation(args=args, train_func=chemprop.train.run_training)